## Assignment Three

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

CAR_DF = 'raw_car_dataset.csv'

df = pd.read_csv(CAR_DF)


# === INITIAL SNAPSHOT ===

In [2]:
print("\n=== INITIAL HEAD ===")
print(df.head(10))


=== INITIAL HEAD ===
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


## TASKS - START

### TASK ONE

##### SHAPE OF DATA

In [3]:
print(df.shape)

(145, 6)


##### INFO OF DATA

In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    str    
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    str    
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 6.9 KB
None


##### MISSING COUNTS

In [5]:
print(df.isnull().sum())

Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


##### LOCATION VALUE COUNTS

In [6]:
print(df["Location"].value_counts(dropna=False))

Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


### TASK TWO

##### PRICE Column remove comma and dollar sign

In [7]:
df["Price"] = df["Price"].replace(r"[\$,]", "", regex=True).astype(float)

print(df["Price"])

0      1500.0
1      4171.0
2      5331.0
3      1500.0
4      1500.0
        ...  
140    1500.0
141    1500.0
142    1500.0
143    1500.0
144    1500.0
Name: Price, Length: 145, dtype: float64


In [8]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    float64
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    str    
 5   Year         145 non-null    int64  
dtypes: float64(3), int64(2), str(1)
memory usage: 6.9 KB
None


##### Print skewness

In [9]:
price_skew = df["Price"].skew()

print(f"Skewness of Price: {price_skew}")

Skewness of Price: 7.871113660850301


### TASK THREE

##### Normalize Location text and map typos (e.g., Subrb → Suburb).

In [10]:
df["Location"] = df["Location"].replace("Subrb", "Suburb")

print(df["Location"].value_counts(dropna=False))


Location
City      59
Suburb    53
Rural     21
??         7
NaN        5
Name: count, dtype: int64


#### Convert Unknowns

In [11]:
df["Location"] = df["Location"].replace("??", pd.NA)

print(df["Location"].value_counts(dropna=False))


Location
City      59
Suburb    53
Rural     21
NaN       12
Name: count, dtype: int64


### TASK FOUR

In [12]:
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())

print(df["Odometer_km"])

0      137830.0
1      128548.0
2      107302.0
3      141838.0
4      128548.0
         ...   
140    223193.0
141    124567.0
142    203153.0
143    180030.0
144    211171.0
Name: Odometer_km, Length: 145, dtype: float64


In [13]:
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])

print(df["Doors"])

0      4.0
1      4.0
2      4.0
3      4.0
4      3.0
      ... 
140    3.0
141    4.0
142    4.0
143    4.0
144    4.0
Name: Doors, Length: 145, dtype: float64


In [14]:
df["Accidents"] = df["Accidents"].fillna(df["Accidents"].mode()[0])

print(df["Accidents"])

0      1
1      0
2      0
3      1
4      0
      ..
140    0
141    2
142    0
143    1
144    0
Name: Accidents, Length: 145, dtype: int64


In [15]:
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

print(df["Location"])


0        City
1       Rural
2      Suburb
3      Suburb
4        City
        ...  
140      City
141    Suburb
142    Suburb
143      City
144      City
Name: Location, Length: 145, dtype: str


In [16]:
print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


### TASK FIVE

##### REMOVE DUPLICATES

In [17]:
print("REPORT SHAPE BEFORE AND AFTER DUPLICATE DROP\n=============================================")
shpe_before = df.shape

df = df.drop_duplicates()

shpe_after = df.shape

print(f"Dropped duplicates: {shpe_before} → {shpe_after}")

REPORT SHAPE BEFORE AND AFTER DUPLICATE DROP
Dropped duplicates: (145, 6) → (140, 6)


### TASK SIX

In [18]:

def capOutliersIQR(dataframe, column_name, multiplier = 1.5 ):

    # Create a copy so we don't accidentally mutate the original data unsafely
    df_cleaned = dataframe.copy()

    # 1. Calculate Q1, Q3, and IQR
    q1 = dataframe[column_name].quantile(0.25)
    q3 = dataframe[column_name].quantile(0.75)
    iqr = q3 - q1

    # 2. Calculate upper and lower fences
    lower_limit = q1 - (multiplier * iqr)
    upper_limit = q3 + (multiplier * iqr)

    return lower_limit, upper_limit

low_price, high_price = capOutliersIQR(df, "Price")
low_odometer, high_odometer = capOutliersIQR(df, "Odometer_km")

df["Price"] = df["Price"].clip(lower=low_price, upper=high_price)
df["Odometer_km"] = df["Odometer_km"].clip(lower=low_odometer, upper=high_odometer)

print(f"PRICES: LOW {low_price} → HIGH {high_price}")
print("==============================================")
print(f"ODOMETER: LOW {low_odometer} → HIGH {high_odometer}")
print("==============================================")

print(df["Price"].describe())
print("==============================================")
print(df["Odometer_km"].describe())

PRICES: LOW -2984.625 → HIGH 8974.375
ODOMETER: LOW -6642.25 → HIGH 271987.75
count     140.000000
mean     3175.456250
std      2601.848629
min      1500.000000
25%      1500.000000
50%      1500.000000
75%      4489.750000
max      8974.375000
Name: Price, dtype: float64
count       140.000000
mean     130945.403571
std       53815.006935
min        5000.000000
25%       97844.000000
50%      128548.000000
75%      167501.500000
max      271987.750000
Name: Odometer_km, dtype: float64


## TASK SEVEN

In [19]:
df = pd.get_dummies(df, columns=["Location"], drop_first=False, dtype=int)

# print(df)

print(df.head(10))

    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   
5  1500.0     211171.0    4.0          0  2003              1               0   
6  1500.0     222235.0    4.0          2  2004              0               1   
7  1500.0     105068.0    5.0          1  2002              1               0   
8  2291.0      90015.0    4.0          0  2007              0               1   
9  1500.0     125976.0    2.0          0  2002              1               0   

   Location_Suburb  
0                0  
1                0  
2                1  
3                1  
4  

## TASK EIGHT

In [20]:
from datetime import datetime

# Get the current year
CURRENT_YEAR = datetime.now().year

df["Car_Age"] = CURRENT_YEAR - df["Year"]

print(df.head(10))

    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   
5  1500.0     211171.0    4.0          0  2003              1               0   
6  1500.0     222235.0    4.0          2  2004              0               1   
7  1500.0     105068.0    5.0          1  2002              1               0   
8  2291.0      90015.0    4.0          0  2007              0               1   
9  1500.0     125976.0    2.0          0  2002              1               0   

   Location_Suburb  Car_Age  
0                0       28  
1                0       10  
2                1

In [21]:
# Log Price

df["Log_Price"] = np.log1p(df["Price"])

print(df.head())


    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   

   Location_Suburb  Car_Age  Log_Price  
0                0       28   7.313887  
1                0       10   8.336151  
2                1       12   8.581482  
3                1       27   7.313887  
4                0        4   7.313887  


In [22]:
# Calculate KM per year safely

# print(df.head(10))

df['Km_Per_Year'] = np.where(
    df["Car_Age"] > 0,
    df["Odometer_km"] / df["Car_Age"],
    df["Odometer_km"]
).astype(float).round(2)

# print(df.info())

df[["Odometer_km", "Car_Age", "Km_Per_Year"]]

,Odometer_km,Car_Age,Km_Per_Year
0,137830.0,28,4922.50
1,128548.0,10,12854.80
2,107302.0,12,8941.83
3,141838.0,27,5253.26
4,128548.0,4,32137.00
...,...,...,...
135,170466.0,12,14205.50
136,219401.0,14,15671.50
137,223338.0,3,74446.00
138,168376.0,18,9354.22


In [23]:
df["Is_Urban"] = df["Location_City"].astype(int)

In [24]:
df.head(10)

,Price,Odometer_km,Doors,Accidents,Year,Location_City,Location_Rural,Location_Suburb,Car_Age,Log_Price,Km_Per_Year,Is_Urban
0,1500.0,137830.0,4.0,1,1998,1,0,0,28,7.313887,4922.50,1
1,4171.0,128548.0,4.0,0,2016,0,1,0,10,8.336151,12854.80,0
2,5331.0,107302.0,4.0,0,2014,0,0,1,12,8.581482,8941.83,0
3,1500.0,141838.0,4.0,1,1999,0,0,1,27,7.313887,5253.26,0
4,1500.0,128548.0,3.0,0,2022,1,0,0,4,7.313887,32137.00,1
5,1500.0,211171.0,4.0,0,2003,1,0,0,23,7.313887,9181.35,1
6,1500.0,222235.0,4.0,2,2004,0,1,0,22,7.313887,10101.59,0
7,1500.0,105068.0,5.0,1,2002,1,0,0,24,7.313887,4377.83,1
8,2291.0,90015.0,4.0,0,2007,0,1,0,19,7.737180,4737.63,0
9,1500.0,125976.0,2.0,0,2002,1,0,0,24,7.313887,5249.00,1


## TASK NINE

In [25]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.to_list()


# print(numeric_cols)


dont_use = {"Price", "Log_Price", "Is_Urban"}

locs = {"Location_"}

num_to_sclale = []

for f in numeric_cols:
    if (f in dont_use or f.startswith("Location_")):
        continue
    num_to_sclale.append(f)

print(num_to_sclale)

scaler = StandardScaler()

df[num_to_sclale] = scaler.fit_transform(df[num_to_sclale])

print(df.head(10))



['Odometer_km', 'Doors', 'Accidents', 'Year', 'Car_Age', 'Km_Per_Year']
    Price  Odometer_km     Doors  Accidents      Year  Location_City  \
0  1500.0     0.128390  0.254091   0.316968 -1.686714              1   
1  4171.0    -0.044709  0.254091  -0.820867  0.794617              0   
2  5331.0    -0.440923  0.254091  -0.820867  0.518913              0   
3  1500.0     0.203135  0.254091   0.316968 -1.548862              0   
4  1500.0    -0.044709 -0.931668  -0.820867  1.621727              1   
5  1500.0     1.496119  0.254091  -0.820867 -0.997456              1   
6  1500.0     1.702451  0.254091   1.454804 -0.859604              0   
7  1500.0    -0.482585  1.439851   0.316968 -1.135307              1   
8  2291.0    -0.763307  0.254091  -0.820867 -0.446049              0   
9  1500.0    -0.092674 -2.117428  -0.820867 -1.135307              1   

   Location_Rural  Location_Suburb   Car_Age  Log_Price  Km_Per_Year  Is_Urban  
0               0                0  1.686714   7.31388

In [26]:
df.head(10)


,Price,Odometer_km,Doors,Accidents,Year,Location_City,Location_Rural,Location_Suburb,Car_Age,Log_Price,Km_Per_Year,Is_Urban
0,1500.0,0.128390,0.254091,0.316968,-1.686714,1,0,0,1.686714,7.313887,-0.615631,1
1,4171.0,-0.044709,0.254091,-0.820867,0.794617,0,1,0,-0.794617,8.336151,0.070446,0
2,5331.0,-0.440923,0.254091,-0.820867,0.518913,0,0,1,-0.518913,8.581482,-0.267993,0
3,1500.0,0.203135,0.254091,0.316968,-1.548862,0,0,1,1.548862,7.313887,-0.587023,0
4,1500.0,-0.044709,-0.931668,-0.820867,1.621727,1,0,0,-1.621727,7.313887,1.738196,1
5,1500.0,1.496119,0.254091,-0.820867,-0.997456,1,0,0,0.997456,7.313887,-0.247276,1
6,1500.0,1.702451,0.254091,1.454804,-0.859604,0,1,0,0.859604,7.313887,-0.167683,0
7,1500.0,-0.482585,1.439851,0.316968,-1.135307,1,0,0,1.135307,7.313887,-0.662741,1
8,2291.0,-0.763307,0.254091,-0.820867,-0.446049,0,1,0,0.446049,7.737180,-0.631621,0
9,1500.0,-0.092674,-2.117428,-0.820867,-1.135307,1,0,0,1.135307,7.313887,-0.587392,1


In [41]:
df.info()

print("===========================================")

print(df.shape)

print("===========================================")

print(df.describe())

print("===========================================")
print("NULL VALUE COUNTS IN DATAFRAME")
print("===========================================")

print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_City    140 non-null    int64  
 6   Location_Rural   140 non-null    int64  
 7   Location_Suburb  140 non-null    int64  
 8   Car_Age          140 non-null    float64
 9   Log_Price        140 non-null    float64
 10  Km_Per_Year      140 non-null    float64
 11  Is_Urban         140 non-null    int64  
dtypes: float64(8), int64(4)
memory usage: 13.3 KB
(140, 12)
             Price   Odometer_km         Doors     Accidents          Year  \
count   140.000000  1.400000e+02  1.400000e+02  1.400000e+02  1.400000e+02   
mean   3175.456250  3.172066e-18  2.077703e-1

In [27]:
OUT_PATH = "car_l3_clean_ready.csv"

df.to_csv(OUT_PATH)

In [46]:
print(pd.__version__)
print(np.__version__)
import sklearn
print(sklearn.__version__)

3.0.3
2.5.0
1.9.0
